
# Outlier-Based Biomarker Discovery for Pregnancy Complications
 Objective: Identify blood, urine, or gut/vaginal microbiome analytes that show extreme deviation in pregnancy complications relative to term controls,  prioritizing these with presistent elevation across multiple timepoints for potential diagnostic biomarker development. 


### Step 1: Calculate Reference Distribution (per analyte)
- for each tissue x timepoint x data type combination
#### 1.1 Subset to term controls only
#### 1.2 Calculate per-analyte statistics
   - Median
   - median avsolute deviation 
       - if control_MAD = 0, flag as "zero_variance" in log and exclude from outlier analysis
#### 1.3 Document control statistics
   - Create reference table with columns as tissue, timepoint, datatype, analyte ID, control median, control_MAD
   - output: control_reference_statistics_tissue_timepoint_datatype.csv

In [83]:
import pandas as pd 
from scipy import stats

In [84]:
# not all timepoint files are present in all batches!
a1 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_110123_A.csv", index_col=0)
a2 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_51223_A.csv", index_col=0)
a3 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_112524_A.csv", index_col=0)


In [85]:
controls = {}
controls["a"] = pd.concat([a1.loc[list(a1["group"] == "Control"),:].drop(columns="group"), a2.loc[list(a2["group"] == "Control"),:].drop(columns="group"), a3.loc[list(a3["group"] == "Control"),:].drop(columns="group")])


In [87]:
a1

,"[Similar to: N1-(4,5-dihydro-1,3-thiazol-2-yl)-2-(2,4-dichlorophenoxy)propanamide; ΔMass: -136.0195 Da]_NEG",p700_POS,[Similar to: Threonine; ΔMass: 83.9612 Da]_NEG,p5160_POS,p1435_POS,p1779_POS,"3,4-Dihydroxybenzylamine_POS",n3923_NEG,p5123_POS,n3638_NEG,...,p5291_POS,3-Hydroxypyridine_POS,2-Hydroxy-4-oxopentanoic acid_NEG,n3808_NEG,n2640_NEG,p4918_POS,group,subgroup,gestational_age,gestational_age_at_sample_collection
DP3-0113A,20.914489,20.971497,17.084475,11.218653,21.708991,20.257634,17.117019,9.647773,13.395766,15.769416,...,9.385259,17.372978,21.869429,9.664750,15.534305,15.404885,"Sample ID\nDP3-0113A Control\nName: group, ...",Sample ID\nDP3-0113A Control\nName: subgrou...,Sample ID\nDP3-0113A 41.5714\nName: gest ag...,Sample ID\nDP3-0113A 10.857143\nName: sampl...
DP3-0179A,19.328288,20.688361,18.262529,11.348922,21.913011,20.651950,17.235937,9.661167,13.455129,16.248774,...,9.438990,17.774454,22.029407,9.595250,15.161603,15.822481,"Sample ID\nDP3-0179A Control\nName: group, ...",Sample ID\nDP3-0179A Control\nName: subgrou...,Sample ID\nDP3-0179A 39.571429\nName: gest ...,Sample ID\nDP3-0179A 12.571429\nName: sampl...
DP3-0175A,20.550679,20.334013,17.487555,11.272921,21.641198,20.637793,17.663977,9.721074,13.475626,15.925020,...,9.380837,21.277266,21.773566,9.637912,15.624616,15.482908,"Sample ID\nDP3-0175A Control\nName: group, ...",Sample ID\nDP3-0175A Control\nName: subgrou...,Sample ID\nDP3-0175A 42.0\nName: gest age d...,Sample ID\nDP3-0175A 11.714286\nName: sampl...
DP3-0265A,20.641966,21.108669,17.723028,11.271574,21.773213,20.464946,17.426202,9.699812,13.440597,16.037519,...,9.379853,17.296159,21.862132,9.583484,15.577669,15.584420,"Sample ID\nDP3-0265A sPTB\nName: group, dty...","Sample ID\nDP3-0265A sPTB\nName: subgroup, ...",Sample ID\nDP3-0265A 28.0\nName: gest age d...,Sample ID\nDP3-0265A 12.571429\nName: sampl...
DP3-0049E A,20.865198,21.170164,17.645873,11.269260,21.941795,20.477195,17.642696,9.788607,13.288281,16.566706,...,9.475084,17.370134,21.842820,9.651322,15.144564,15.903328,"Sample ID\nDP3-0049E A FGR\nName: group, dt...",Sample ID\nDP3-0049E A FGR <3\nName: subgro...,Sample ID\nDP3-0049E A 36.428571\nName: ges...,Sample ID\nDP3-0049E A 10.571429\nName: sam...
DP3-0142A,20.428059,20.819311,17.760235,11.391077,22.211055,20.338701,17.490428,9.721783,13.464209,15.733074,...,9.410929,17.470621,21.580563,9.534804,16.774911,15.916290,"Sample ID\nDP3-0142A Control\nName: group, ...",Sample ID\nDP3-0142A Control\nName: subgrou...,Sample ID\nDP3-0142A 37.714286\nName: gest ...,Sample ID\nDP3-0142A 13.142857\nName: sampl...
DP3-0062A,21.389806,20.879905,16.058104,11.208881,21.836735,20.375320,17.386091,9.759498,13.324196,16.117926,...,9.380780,17.581508,21.979079,9.591293,15.261859,15.277738,"Sample ID\nDP3-0062A Control\nName: group, ...",Sample ID\nDP3-0062A Control\nName: subgrou...,Sample ID\nDP3-0062A 39.0\nName: gest age d...,Sample ID\nDP3-0062A 9.714286\nName: sample...
DP3-0006A,20.843003,21.097012,17.814144,11.329767,22.200921,20.622683,15.528038,9.781601,13.308522,17.015661,...,9.404667,17.786068,22.128504,9.576932,16.493824,15.304600,"Series([], Name: group, dtype: str)","Series([], Name: subgroup, dtype: str)","Series([], Name: gest age del, dtype: float64)","Series([], Name: sample gest Age, dtype: float64)"
DP3-0070A,20.656274,21.454947,17.523143,11.256103,22.001294,20.426561,17.618280,9.826980,13.237746,16.030709,...,9.381549,18.133304,21.698995,9.601442,15.953398,15.552664,"Sample ID\nDP3-0070A Control\nName: group, ...",Sample ID\nDP3-0070A Control\nName: subgrou...,Sample ID\nDP3-0070A 39.571429\nName: gest ...,Sample ID\nDP3-0070A 8.285714\nName: sample...
DP3-0235A,20.730050,20.862584,17.583292,11.246132,21.922917,20.501010,16.960174,9.758822,13.416750,15.445089,...,9.339468,18.565931,21.996408,9.581320,14.889790,15.599034,"Sample ID\nDP3-0235A FGR\nName: group, dtyp...",Sample ID\nDP3-0235A FGR <5\nName: subgroup...,Sample ID\nDP3-0235A 39.571429\nName: gest ...,Sample ID\nDP3-02

In [4]:
a_med = controls["a"].median()

In [5]:
a_mad = stats.median_abs_deviation(controls["a"])


In [6]:
x = pd.DataFrame(index=a_med.index)

In [7]:
x["med"] = a_med
x["mad"] = a_mad
x["tissue"] = "plasma"
x["timepoint"] = "A"
x["datatype"] = "metabolites"
x["analyte_ID"] = a_med.index

In [8]:
x

,med,mad,tissue,timepoint,datatype,analyte_ID
Amoxicillin_POS,13.341908,0.135476,plasma,A,metabolites,Amoxicillin_POS
p487_POS,23.117087,0.465887,plasma,A,metabolites,p487_POS
n304_NEG,22.091415,0.258679,plasma,A,metabolites,n304_NEG
p3014_POS,19.410243,0.356406,plasma,A,metabolites,p3014_POS
n694_NEG,20.831865,0.153171,plasma,A,metabolites,n694_NEG
...,...,...,...,...,...,...
p5751_POS,15.351127,0.724511,plasma,A,metabolites,p5751_POS
p3224_POS,19.309842,0.385820,plasma,A,metabolites,p3224_POS
p3083_POS,18.495527,0.375046,plasma,A,metabolites,p3083_POS
p411_POS,24.532796,0.246376,plasma,A,metabolites,p411_POS


In [9]:
plasma_110123 = pd.DataFrame(columns=["tissue", "timepoint", "datatype", "analyte_ID", "control_median", "control_MAD"])
# tissue, timepoint, data type, analyte ID, control median, control MAD

In [10]:
timepoints = ["a", "b", "c", "d", "e"]
for t in timepoints:
    med = controls[t].median()
    mad = stats.median_abs_deviation(controls[t])
    temp = pd.DataFrame()
    temp["control_median"] = med
    temp["control_MAD"] = mad
    temp["tissue"] = "plasma"
    temp["timepoint"] = t
    temp["datatype"] = "metabolites"
    temp["analyte_ID"] = med.index
    temp.to_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/control_reference_statistics_plasma_" + t + "_MTBL.csv")
    

KeyError: 'b'

In [11]:
batches = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/pos_batch.csv")["batch"].unique()

In [16]:
batches = batches.tolist()

In [13]:
x["mad"] == 0

Amoxicillin_POS    False
p487_POS           False
n304_NEG           False
p3014_POS          False
n694_NEG           False
                   ...  
p5751_POS          False
p3224_POS          False
p3083_POS          False
p411_POS           False
p4768_POS          False
Name: mad, Length: 1885, dtype: bool

In [ ]:
# Step 2: Calculate MAD Scores (All Samples)
# For each sample in each tissue at each timepoint for each data type:
# 2.1 Calculate MAD Score
#   For each analyte measurement:
#       MAD_score = (observed_value - Control_Median) / Control_MAD
#   Use the control_median and control_MAD from the appropriate tissue/timepoint/data_type combination
# 2.2 Create MAD score matrix
#   Generate a data structure with:
#       Rows = Sample_ID
#       Columns = Analytes
#       Values = MAD scores
#       Additional metadata = patient_ID, group (control/FGR/GHDP/sPTB), group_subtype, gestational_age, gestational_age_at_sample_collection
# Output: mad_scores_matrix_<tissue>_<timepoint>_<data_type>.csv

In [15]:
def mergeBatches(batches, dir_input, t):
    temp = pd.DataFrame()
    for b in batches:
        try:
            samples = pd.read_csv(dir_input + "/Samples_" + str(b) + "_" + t + ".csv", index_col=0)
            temp = pd.concat([temp, samples])
        except:
            continue
    return temp

In [17]:
allSamples = mergeBatches(batches,"/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma","A")

In [18]:
allSamples

,Amoxicillin_POS,p487_POS,n304_NEG,p3014_POS,n694_NEG,p1103_POS,p1187_POS,n3543_NEG,p4718_POS,p5146_POS,...,p952_POS,p4677_POS,p2942_POS,p3713_POS,p5751_POS,p3224_POS,p3083_POS,p411_POS,p4768_POS,group
DP3-0048 A,9379.806338,8.247729e+06,5.511467e+06,504502.131659,1.314213e+06,32065.899253,5.648079e+05,4181.140011,15221.778478,10101.863297,...,39308.767055,41564.676367,23049.996905,291345.658827,20425.264764,502958.157223,256820.296052,2.428852e+07,8419.803775,HDP
DP3-0071 A,10060.318594,1.576856e+07,5.500333e+06,513918.001449,2.074990e+06,30696.796666,8.546949e+05,4229.895221,16650.206520,14251.174880,...,37630.419155,57226.672027,24995.112636,321743.976335,71858.526761,554260.623233,329584.460759,2.887673e+07,9997.300236,HDP
DP3-0124 A,9433.282152,9.413858e+06,5.001819e+06,669182.170262,1.869225e+06,45337.662878,6.211752e+05,4169.111703,12709.441191,12421.539989,...,55578.283173,74733.934524,27738.148263,347794.685307,66974.684732,558898.355054,537888.612354,2.571056e+07,15883.942624,HDP
DP3-0018 A,9403.233411,7.869484e+06,3.642611e+06,671964.055024,2.706375e+06,27679.336535,1.076997e+06,4180.040128,14626.333085,12180.649750,...,33931.391835,39712.790549,28366.559932,293692.633536,33537.960640,552280.112865,694828.794380,2.926493e+07,4500.272941,Control
DP3-0037 A,9396.326803,1.092705e+07,5.179204e+06,666625.256181,2.317996e+06,34660.188077,6.089777e+05,4273.775624,13927.333148,14819.419821,...,42489.039476,61417.172339,28850.631560,396264.534735,40608.817536,518867.803673,101848.578155,2.675625e+07,11013.540980,Control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DP3-0259A,13.158357,2.225081e+01,2.202200e+01,19.520956,2.062286e+01,15.230781,1.950810e+01,12.057363,13.927124,13.704948,...,15.563555,16.230901,14.618236,18.254694,14.462860,19.449606,18.113915,2.470307e+01,12.906893,FGR
DP3-0387A,13.094710,2.305691e+01,2.218167e+01,18.982250,2.072640e+01,15.139903,1.943932e+01,12.094264,14.148986,13.755664,...,15.411007,15.366876,14.649988,16.613437,15.156855,18.821212,18.168661,2.447427e+01,13.598765,Control
DP3-0279A,13.461306,2.316644e+01,2.218118e+01,19.185503,2.078922e+01,15.283544,1.912035e+01,17.076084,13.526867,14.223920,...,15.589389,16.729329,14.767952,19.168781,15.252523,19.309842,17.488428,2.446218e+01,13.143739,Control
DP3-0392A,13.313893,2.334989e+01,2.220748e+01,19.589938,2.076556e+01,15.041650,1.986793e+01,11.910799,17.237419,13.827380,...,15.405164,16.699702,14.766655,18.629943,15.496318,19.515361,18.439698,2.442961e+01,13.385253,HDP


In [19]:
x

,med,mad,tissue,timepoint,datatype,analyte_ID
Amoxicillin_POS,13.341908,0.135476,plasma,A,metabolites,Amoxicillin_POS
p487_POS,23.117087,0.465887,plasma,A,metabolites,p487_POS
n304_NEG,22.091415,0.258679,plasma,A,metabolites,n304_NEG
p3014_POS,19.410243,0.356406,plasma,A,metabolites,p3014_POS
n694_NEG,20.831865,0.153171,plasma,A,metabolites,n694_NEG
...,...,...,...,...,...,...
p5751_POS,15.351127,0.724511,plasma,A,metabolites,p5751_POS
p3224_POS,19.309842,0.385820,plasma,A,metabolites,p3224_POS
p3083_POS,18.495527,0.375046,plasma,A,metabolites,p3083_POS
p411_POS,24.532796,0.246376,plasma,A,metabolites,p411_POS


In [ ]:
# remove NaNs
# check if mad is 0, if is, log
scores = pd.DataFrame(index=allSamples.index)

for m in allSamples.columns[:-1]:
    temp = (allSamples[m] - x.loc[m, "med"]) / x.loc[m, "mad"]
    scores[m] = temp

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_51152/1045040521.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[m] = temp


In [31]:
scores

,Amoxicillin_POS,p487_POS,n304_NEG,p3014_POS,n694_NEG,p1103_POS,p1187_POS,n3543_NEG,p4718_POS,p5146_POS,...,p469_POS,p952_POS,p4677_POS,p2942_POS,p3713_POS,p5751_POS,p3224_POS,p3083_POS,p411_POS,p4768_POS
DP3-0048 A,69137.301165,1.770321e+07,2.130609e+07,1.415473e+06,8.579899e+06,185647.988824,1.795936e+06,68817.999792,31658.522557,30771.964180,...,3.705260e+06,199070.475378,70524.492119,205653.751491,301328.356437,28170.604684,1.303558e+06,6.847211e+05,9.858317e+07,4988.317598
DP3-0071 A,74160.411132,3.384624e+07,2.126305e+07,1.441892e+06,1.354674e+07,177717.707797,2.717730e+06,69622.790413,34632.070015,43428.915122,...,3.300950e+06,190567.502706,97109.048276,223019.374646,332770.208288,99160.914447,1.436528e+06,8.787352e+05,1.172060e+08,5924.440005
DP3-0124 A,69532.025718,2.020624e+07,1.933590e+07,1.877530e+06,1.220338e+07,262522.297800,1.975175e+06,68619.451384,26428.609720,37847.844962,...,2.919169e+06,281496.324619,126825.746782,247508.674824,359715.203909,92420.034226,1.448548e+06,1.434145e+06,1.043550e+08,9417.707965
DP3-0018 A,69310.224965,1.689133e+07,1.408149e+07,1.885336e+06,1.766883e+07,160239.611515,3.424615e+06,68799.844279,30418.988550,37113.039712,...,3.352906e+06,171827.211594,67381.114762,253119.014640,303755.899708,46269.288413,1.431394e+06,1.852601e+06,1.187816e+08,2662.378535
DP3-0037 A,69259.244763,2.345421e+07,2.002163e+07,1.870356e+06,1.513324e+07,200674.941252,1.936389e+06,70347.113760,28963.885860,45162.274549,...,3.021885e+06,215182.611302,104221.972663,257440.714032,409848.959652,56028.776765,1.344794e+06,2.715137e+05,1.085993e+08,6527.500459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DP3-0259A,-1.354862,-1.859422e+00,-2.683438e-01,3.106387e-01,-1.364522e+00,0.243680,-9.021799e-02,-0.052872,0.457598,-0.688774,...,-2.949984e+00,0.285187,0.466589,-1.726435,0.064879,-1.226023,3.622515e-01,-1.017507e+00,6.911198e-01,-0.526905
DP3-0387A,-1.824665,-1.291736e-01,3.489109e-01,-1.200858e+00,-6.885499e-01,-0.282712,-3.089097e-01,0.556242,0.919446,-0.534070,...,-6.378729e-01,-0.487663,-1.000000,-1.442962,-1.632721,-0.268143,-1.266469e+00,-8.715357e-01,-2.375504e-01,-0.116333
DP3-0279A,0.881315,1.059300e-01,3.470259e-01,-6.305717e-01,-2.784036e-01,0.549304,-1.323203e+00,82.789969,-0.375614,0.894286,...,-2.795542e+00,0.416070,1.312618,-0.389800,1.010345,-0.136098,0.000000e+00,-2.685270e+00,-2.866353e-01,-0.386355
DP3-0392A,-0.206791,4.996994e-01,4.486941e-01,5.041866e-01,-4.328502e-01,-0.851826,1.053990e+00,-2.472157,7.348614,-0.315311,...,1.651954e-01,-0.517263,1.262330,-0.401374,0.453009,0.200399,5.326817e-01,-1.488595e-01,-4.188225e-01,-0.243035


In [88]:
meta = pd.read_excel("/Users/kaylaxu/Desktop/data/raw_data/dp3 master table v2.xlsx", index_col=0, sheet_name="n=133 metabolomics")

In [89]:
clean_meta = meta[meta.index.notna()]

In [100]:
list(clean_meta.loc[clean_meta["Sample ID"] == "DP3-0165EA", "group"])[0]

'HDP'